# 📘 AI/ML on Databricks & Agentic Patterns
## MLflow, Feature Store, Model Serving & AI Agents

**What you'll learn:**
- ML lifecycle on Databricks (train, track, deploy, monitor)
- MLflow for experiment tracking and model registry
- Feature Store for ML feature management
- Model Serving (real-time inference endpoints)
- Vector Search for RAG applications
- AI agents and LLM integration
- Data engineering for ML pipelines

**Prerequisites:** Notebooks 03 (PySpark), 04 (Medallion)
**Study Order:** 40

---

---
# 📗 Section 1: ML Lifecycle on Databricks

## The ML Pipeline (Data Engineer's Perspective)

Data engineers build the FOUNDATION that ML depends on:

```
DATA ENGINEER'S WORK:                    ML ENGINEER'S WORK:
┌─────────────────────────┐             ┌─────────────────────────┐
│ 1. Ingest raw data      │             │ 5. Train models         │
│ 2. Clean & transform    │────────────▶│ 6. Evaluate & compare   │
│ 3. Build features       │             │ 7. Deploy to serving    │
│ 4. Serve to Feature Store│            │ 8. Monitor in production│
└─────────────────────────┘             └─────────────────────────┘
```

## Key ML Components on Databricks

| Component | What It Does | DE Responsibility |
|-----------|-------------|-------------------|
| **MLflow** | Track experiments, register models | Set up tracking server, log metrics |
| **Feature Store** | Manage ML features | Build feature pipelines, ensure freshness |
| **Model Serving** | Real-time inference | Ensure serving infrastructure, monitor |
| **Vector Search** | Similarity search for RAG | Build embedding pipelines |
| **Unity Catalog** | Govern models and features | Permissions, lineage, audit |

## MLflow — Experiment Tracking

```python
import mlflow

# Track an experiment
with mlflow.start_run():
    # Log parameters
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("n_estimators", 100)
    
    # Train model
    model = train_model(X_train, y_train)
    
    # Log metrics
    mlflow.log_metric("accuracy", 0.95)
    mlflow.log_metric("f1_score", 0.93)
    
    # Log model
    mlflow.sklearn.log_model(model, "model")
```

## Feature Store

```python
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()

# Create feature table
fs.create_table(
    name="prod.ml_features.customer_features",
    primary_keys=["customer_id"],
    df=customer_features_df,
    description="Customer features for churn prediction"
)

# Read features for training
training_set = fs.create_training_set(
    df=labels_df,
    feature_lookups=[
        FeatureLookup(table_name="prod.ml_features.customer_features",
                     lookup_key="customer_id")
    ]
)
```

In [ ]:
# Simulating ML pipeline from a DE perspective
class MLPipelineForDE:
    """What data engineers build to support ML teams."""
    
    def __init__(self):
        self.feature_tables = {}
        self.freshness_slas = {}
    
    def build_feature_pipeline(self, name, source_tables, features, refresh_schedule):
        """Build a feature engineering pipeline."""
        self.feature_tables[name] = {
            "sources": source_tables,
            "features": features,
            "refresh": refresh_schedule,
            "status": "active"
        }
        self.freshness_slas[name] = refresh_schedule
    
    def check_freshness(self):
        """Monitor feature freshness (DE responsibility)."""
        print("📊 FEATURE FRESHNESS MONITOR")
        print("=" * 60)
        for name, schedule in self.freshness_slas.items():
            print(f"   ✅ {name}: refreshed {schedule}")
    
    def show_pipeline(self):
        """Show all feature pipelines."""
        print("🔧 ML FEATURE PIPELINES (built by Data Engineers)")
        print("=" * 60)
        for name, details in self.feature_tables.items():
            print(f"\n   📌 {name}")
            print(f"      Sources: {details['sources']}")
            print(f"      Features: {details['features'][:3]}...")
            print(f"      Refresh: {details['refresh']}")


# Demo
ml_pipeline = MLPipelineForDE()

ml_pipeline.build_feature_pipeline(
    "customer_churn_features",
    source_tables=["silver.orders", "silver.customers", "silver.support_tickets"],
    features=["total_orders_30d", "avg_order_value", "days_since_last_order",
              "support_tickets_count", "lifetime_value", "payment_failures"],
    refresh_schedule="every 4 hours"
)

ml_pipeline.build_feature_pipeline(
    "fraud_detection_features",
    source_tables=["silver.transactions", "silver.devices", "silver.locations"],
    features=["transaction_velocity_1h", "new_device_flag", "distance_from_home",
              "amount_vs_average", "time_since_last_transaction"],
    refresh_schedule="every 5 minutes (near real-time)"
)

ml_pipeline.show_pipeline()
ml_pipeline.check_freshness()

print("\n💡 DE Role in ML:")
print("   • Build reliable feature pipelines (freshness SLAs)")
print("   • Ensure data quality for training data")
print("   • Manage feature store infrastructure")
print("   • Monitor serving endpoints (latency, errors)")


---
# 📗 Section 2: ML Lifecycle on Databricks

## The Data Engineer's Role in ML

Data engineers build the FOUNDATION that ML depends on:

```
DATA ENGINEER:                          ML ENGINEER:
1. Ingest raw data                      5. Train models
2. Clean and transform                  6. Evaluate and compare
3. Build features          ---------->  7. Deploy to serving
4. Serve to Feature Store               8. Monitor in production
```

## MLflow -- Experiment Tracking

```python
import mlflow

with mlflow.start_run():
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("n_estimators", 100)
    
    model = train_model(X_train, y_train)
    
    mlflow.log_metric("accuracy", 0.95)
    mlflow.log_metric("f1_score", 0.93)
    mlflow.sklearn.log_model(model, "model")
```

## Feature Store

```python
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()

# Create feature table
fs.create_table(
    name="prod.ml_features.customer_features",
    primary_keys=["customer_id"],
    df=customer_features_df,
    description="Customer features for churn prediction"
)

# Read features for training
training_set = fs.create_training_set(
    df=labels_df,
    feature_lookups=[
        FeatureLookup(
            table_name="prod.ml_features.customer_features",
            lookup_key="customer_id"
        )
    ]
)
```

## Vector Search for RAG

```python
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# Create vector search index
vsc.create_delta_sync_index(
    endpoint_name="my-endpoint",
    index_name="prod.docs.embeddings_index",
    source_table_name="prod.docs.documents",
    pipeline_type="TRIGGERED",
    primary_key="doc_id",
    embedding_source_column="content",
    embedding_model_endpoint_name="databricks-bge-large-en"
)

# Query similar documents
results = vsc.get_index("prod.docs.embeddings_index").similarity_search(
    query_text="How do I optimize Spark jobs?",
    num_results=5
)
```

In [ ]:
# ML pipeline from a DE perspective
print("ML Pipeline -- Data Engineer Responsibilities")
print("=" * 60)

ml_de_responsibilities = {
    "Feature Engineering": {
        "tasks": [
            "Build feature pipelines (Spark jobs)",
            "Ensure feature freshness (SLAs)",
            "Register features in Feature Store",
            "Monitor feature drift over time",
        ],
        "tools": ["PySpark", "Databricks Feature Store", "MLflow"],
    },
    "Training Data": {
        "tasks": [
            "Create training datasets (point-in-time correct)",
            "Handle class imbalance",
            "Split train/validation/test properly",
            "Version training datasets",
        ],
        "tools": ["Delta Lake", "MLflow", "PySpark"],
    },
    "Model Serving Infrastructure": {
        "tasks": [
            "Set up serving endpoints",
            "Monitor prediction latency",
            "Log predictions for monitoring",
            "Handle model versioning",
        ],
        "tools": ["Databricks Model Serving", "MLflow Registry"],
    },
    "Data Quality for ML": {
        "tasks": [
            "Detect feature drift (distribution changes)",
            "Monitor prediction quality over time",
            "Alert on data anomalies",
            "Validate training data quality",
        ],
        "tools": ["Great Expectations", "Evidently", "Custom checks"],
    },
}

for area, details in ml_de_responsibilities.items():
    print(f"\n{area}:")
    print(f"  Tools: {', '.join(details['tools'])}")
    print(f"  Tasks:")
    for task in details["tasks"]:
        print(f"    * {task}")


---
*Notebook 40 of the Databricks Data Engineering series*
*Study Order: 40*

---
*Notebook 40 of the Databricks Data Engineering series*
*Study Order: 40*